# Today’s lunch menus with Tavily Research

Tavily Research handles menu discovery, reading and structured meal generation. There is no OpenAI client or GPT extraction step in this workflow. Python then uses Tavily Extract to retrieve the cited menu sources and check the date and evidence before saving the same app JSON.

**Costs:** Research `mini` uses 4–110 Tavily credits per request, rather than the one credit used by basic Search. Extract verification uses additional credits. This is not guaranteed to be cheaper. No API calls happen until section 3. Start with one restaurant if you want to limit the experiment. Research may still fail to read some images/PDFs; failed verification blocks saving rather than silently publishing old meals.

References: [Research](https://docs.tavily.com/documentation/api-reference/endpoint/research), [Extract](https://docs.tavily.com/documentation/api-reference/endpoint/extract), [credits](https://docs.tavily.com/documentation/api-credits).

## 1. Load Tavily and the helper

Only `TAVILY_API_KEY` is needed in local `.env`. The helper `tavily_agent.py` contains the schema, API calls and source checks so the notebook stays readable. It uses the existing httpx/Pydantic/dotenv packages. Restart the kernel after updating helper code to avoid using an old import.

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

from dotenv import load_dotenv

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from tavily_agent import TavilyMenus, ResearchMenu, save_menus

load_dotenv(project_root / ".env", override=True)
tavily_key = os.getenv("TAVILY_API_KEY")
print("Tavily key loaded:", bool(tavily_key))

agent = TavilyMenus(tavily_key, project_root / "data/tavily-research")


## 2. Select restaurants and date

The full list is retained. Use `restaurants[:1]` instead of `restaurants` for the first Research run. The save cell writes only the selected list, so choose the full list before replacing the app’s full menu. Starting research does not change the existing JSON.

In [ ]:
restaurants = [
    {
        "name": "Spilka",
        "address": "Legionárska 8, Bratislava",
        "website": "https://www.spilkarestaurant.sk/",
        "domain": "spilkarestaurant.sk"
    },
    {
        "name": "Centrálna Klubovňa",
        "address": "Krížna 64, Bratislava",
        "website": "https://www.klubovna.sk/",
        "domain": "klubovna.sk"
    },
    {
        "name": "ALDA",
        "address": "Račianska 35, Bratislava",
        "website": "https://www.aldajedalen.sk/",
        "domain": "aldajedalen.sk"
    },
    {
        "name": "Cafe & Restaurant Michaela",
        "address": "Kýčerského 10, Bratislava",
        "website": "https://www.restauraciamichaela.sk/",
        "domain": "restauraciamichaela.sk"
    }
]

selected_restaurants = restaurants  # Use restaurants[:1] to test Spilka first.
today = datetime.now(ZoneInfo("Europe/Bratislava")).date()
print("Requested date:", today)
print("Restaurants selected:", len(selected_restaurants))


## 3. Start or resume Research tasks

This submits one `mini` research task per selected restaurant. The prompt requires following actual daily-menu links and reading the menu itself, with exact date/day/price evidence in structured output. Research domain preferences are soft; Python separately restricts accepted sources to official pages or assets linked by retrieved official pages.

Task IDs are saved under ignored `data/tavily-research/`. Running again with the same restaurant/date/schema resumes the saved tasks rather than creating duplicate tasks. Changed dates/configuration create new tasks. A submission whose response was lost is marked uncertain and requires checking the Tavily dashboard before retrying. This notebook does not automatically poll or sleep.

In [ ]:
research_tasks = []
for restaurant in selected_restaurants:
    task = agent.start(restaurant, today)
    research_tasks.append(task)
    print(restaurant["name"], task["status"], task["request_id"])


## 4. Check completion and verify retrieved sources

Rerun this cell while tasks are pending; it checks existing task IDs without starting new research. On completion, it uses Tavily Extract once to retrieve up to four cited menu sources plus the official homepage. Raw task results, fetched source text and credit usage are saved locally for diagnosis.

Python requires the quoted full date or week to occur in retrieved source text and cover the requested calendar date. It also checks each meal’s day and price evidence, and rejects zero soup surcharges without explicit inclusion wording. These conservative checks can reject legitimate formats and are not a complete factual or completeness proof. Unsupported image/PDF extraction, unverifiable dates or missing evidence produce `needs_review` and block saving. Completed verified results are reused.

In [ ]:
menu_results = []
for task in research_tasks:
    result = agent.check(task)
    menu_results.append(result)
    print("\n", result["restaurant"]["name"], "—", result["status"])
    if result.get("reason"):
        print("Review:", result["reason"])
    print("Research usage:", result.get("research", {}).get("usage"))
    print("Retrieval usage:", result.get("retrieved", {}).get("usage"))
    print("Meals:", len(result.get("meals", [])))
    for meal in result.get("meals", []):
        print(meal)


## 5. Save verified meals

No API call is made here. All selected restaurants must have verified results for the requested date. Pending, failed, empty or unverified results leave `menus.json` untouched. Saving replaces the file with the selected restaurants’ meals, keeping the existing flat schema, soup options and labels. Review the results before running this cell, then refresh the app.

In [ ]:
menus_data = save_menus(
    menu_results, selected_restaurants, today, project_root / "menus.json"
)
print("Saved", len(menus_data["meals"]), "meals")
